In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/deep-past-initiative-machine-translation/train.csv")
df = df.drop(columns=["oare_id"])
df


In [ ]:
import re

def clean_transliteration(text):

    text = re.sub(r'[!?⁄:—]', '', text)


    text = re.sub(r'(?<!\d)\.(?!\d)', '', text)

    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'˹.*?˺', '', text)
    text = re.sub(r'\[.*?\]', '', text)


    subscript_map = {
        '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
        '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9'
    }
    for sub, normal in subscript_map.items():
        text = text.replace(sub, normal)


    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [ ]:
def replace_gaps(text):
    if pd.isna(text): 
        return text
    
    text = re.sub(r'\[\.{3}(?:\s+\.{3})+\]\s+\.{3}(?:\s+\.{3})+', '<big_gap>', text)
    text = re.sub(r'\[\.{3}(?:\s+\.{3})+\]', '<big_gap>', text)
    text = re.sub(r'\.{3}(?:\s+\.{3})+', '<big_gap>', text)

    text = re.sub(r'\[x\]', '<gap>', text)
    text = re.sub(r' x ', ' <gap> ', text)
    text = re.sub(r'\[…\]', '<big_gap>', text)
    text = re.sub(r'\[\.\.\.\]', '<big_gap>', text)
    text = re.sub(r'…', '<big_gap>', text)
    text = re.sub(r'\.\.\.', '<big_gap>', text)
    text = re.sub(r'(x)+', '<gap>', text)

    return text

In [ ]:
def clean_gaps(text):
  if pd.isna(text): 
        return text
  text = re.sub(r'<gap> x ', ' <gap> ', text)
  text = re.sub(r'x <gap>', ' <gap>', text)
  text = re.sub(r'<gap> <gap>', '<gap>', text)
  text = re.sub(r'(<gap> )+', r'\1', text)
  return text


In [ ]:
df['cleaned_akkadian'] = df['transliteration'].apply(clean_transliteration)
df['cleaned_akkadian'] = df['cleaned_akkadian'].apply(replace_gaps)
df['cleaned_akkadian'] = df['cleaned_akkadian'].apply(clean_gaps)

df = df.drop(columns='transliteration')

df



In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

dataset = Dataset.from_pandas(df)
split_datasets = dataset.train_test_split(test_size=0.1, seed=42)


**Get 4 Cell down if you dont want to see training code**

In [ ]:
from transformers import ByT5Tokenizer, T5ForConditionalGeneration

MODEL_NAME = "google/byt5-small"

tokenizer = ByT5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.decoder_start_token_id = tokenizer.pad_token_id
model.config.use_cache = False

In [ ]:
PREFIX = "translate Akkadian to English: "

def preprocess_function(examples):

    return tokenizer(
        examples["cleaned_akkadian"],
        max_length=512,
        truncation=True,
        text_target=examples["translation"],
    )



tokenized_train = split_datasets["train"].map(preprocess_function, batched=True)
tokenized_val = split_datasets["test"].map(preprocess_function, batched=True)

In [ ]:
from transformers import Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer

args = Seq2SeqTrainingArguments(
    output_dir="./byt5_akkadian",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    fp16=False,              
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,    
    weight_decay=0.02,
    save_total_limit=2,
    num_train_epochs=20,
    predict_with_generate=True,
    logging_steps=10,
    report_to="none"
)



data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)


trainer.train()


In [ ]:
#This is the same model that we trained above

import kagglehub
path = kagglehub.model_download("adarsh2626/akkadian-to-en-byt5/pyTorch/v1")

In [ ]:
import torch
from transformers import ByT5Tokenizer, T5ForConditionalGeneration
import pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

path = "/kaggle/input/akkadian-to-en-byt5/pytorch/v1/1"

tokenizer = ByT5Tokenizer.from_pretrained(path)
model = T5ForConditionalGeneration.from_pretrained(path).to(DEVICE)
model.eval()

test_df = pd.read_csv("/kaggle/input/deep-past-initiative-machine-translation/test.csv")

START = "translate Akkadian to English: "

test_df['cleaned_akkadian'] = test_df['transliteration'].apply(clean_transliteration)
test_df['cleaned_akkadian'] = test_df['cleaned_akkadian'].apply(replace_gaps)
test_df['cleaned_akkadian'] = test_df['cleaned_akkadian'].apply(clean_gaps)

test_df["cleaned_akkadian"] = [START + i for i in test_df["cleaned_akkadian"]]
test_df

In [ ]:
from torch.utils.data import DataLoader, Dataset

class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['cleaned_akkadian'].astype(str).tolist()
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text, 
            max_length=512, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }

test_dataset = InferenceDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)


In [ ]:
all_predictions = []

model.eval()
with torch.no_grad():
    for batch in test_loader:   
        input_ids = batch["input_ids"].to(DEVICE)        
        attention_mask = batch["attention_mask"].to(DEVICE)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=284,
            num_beams=4,
            repetition_penalty=1.2,          
            early_stopping=True
        )




        decoded = tokenizer.batch_decode(
            outputs, skip_special_tokens=True
        )
        all_predictions.extend([d.strip() for d in decoded])
print(all_predictions)

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": all_predictions
})

submission["translation"] = submission["translation"].apply(lambda x: x if len(x) > 0 else "broken text")

submission.to_csv("submission.csv", index=False)
print("Submission file saved successfully!")
submission.head()

